In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pickle
import os

In [2]:
def train_model(data_path):
    try:
        # Load the preprocessed data
        df = pd.read_csv(data_path)

        # Handle missing values and empty strings (crucial)
        df.dropna(subset=['clean_text'], inplace=True)
        df = df[df['clean_text'] != ""]
        df.reset_index(drop=True, inplace=True)

        X = df['clean_text']
        y = df['label']

        # Split the dataset
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Vectorize text
        vectorizer = TfidfVectorizer(max_features=5000)
        X_train_tfidf = vectorizer.fit_transform(X_train)
        X_test_tfidf = vectorizer.transform(X_test)

        # Train the model
        model = LogisticRegression(max_iter=1000)
        model.fit(X_train_tfidf, y_train)

        # Evaluate the model
        y_pred = model.predict(X_test_tfidf)
        print("Classification Report:")
        print(classification_report(y_test, y_pred))
        print("Confusion Matrix:")
        print(confusion_matrix(y_test, y_pred))
        print("Accuracy:", accuracy_score(y_test, y_pred))

        # Save the model and vectorizer (create directory if it doesn't exist)
        os.makedirs("../models", exist_ok=True)  # Create 'models' directory if it doesn't exist.
        with open("../models/model.pkl", "wb") as f:
            pickle.dump(model, f)
        with open("../models/vectorizer.pkl", "wb") as f:
            pickle.dump(vectorizer, f)
        print("Model and vectorizer saved in the 'models' directory.")

    except FileNotFoundError:
        print(f"Error: File not found at {data_path}")
    except ValueError as e: # Catch value errors, for example if label column is not present
        print(f"ValueError: {e}")
    except Exception as e:
        print(f"An error occurred: {e}")

In [3]:
if __name__ == "__main__":
    train_model("../data/combined_news_preprocessed.csv")

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4567
           1       0.98      0.99      0.99      4287

    accuracy                           0.99      8854
   macro avg       0.99      0.99      0.99      8854
weighted avg       0.99      0.99      0.99      8854

Confusion Matrix:
[[4498   69]
 [  51 4236]]
Accuracy: 0.9864468037045403
Model and vectorizer saved in the 'models' directory.
